# Load Packages

In [1]:
import os
import pandas as pd

# Parameters

In [2]:
# Output directory — all plots and tables land here
output_dir = os.path.join("..", "outputs")
os.makedirs(output_dir, exist_ok=True)

# Data directory — all data files land here
data_raw_dir = os.path.join("..", "data", "raw")
data_processed_dir = os.path.join("..", "data", "processed")

# Load and Pre-Process Data

In [3]:
# Load dataset
toaster_file_path = os.path.join(data_raw_dir, "Param_Toaster Data_2018_23.xlsx")
toaster_df = pd.read_excel(toaster_file_path)

# Data preview
toaster_df.head()

,ASIN,P_TITLE,OP,DP,SP,FS,PRA_4.5,P_RTG,RTG_P_NO,SELLER_LINK,...,RV_DT,VP,HLP_VT,IMG_PRST,TTL_RV,RVS_L,RV_TRANS,SUBJ,SRVS,CP_RVS
0,B009GQ034C,Cuisinart CPT-122 Compact Plastic 2-Slice Toas...,55.462963,0.460000,29.95,1,0,4.3,27270,https://www.amazon.com/stores/Cuisinart/page/9...,...,2018-01-01,1,.,0,6628,24,"Very happy. Thanks, Tom.",0.6,positive,0.765
1,B009GQ034C,Cuisinart CPT-122 Compact Plastic 2-Slice Toas...,55.462963,0.460000,29.95,1,0,4.3,27270,https://www.amazon.com/stores/Cuisinart/page/9...,...,2018-01-01,1,.,0,6628,24,"Very happy. Thanks, Tom.",0.6,positive,0.765
2,B009GQ034C,Cuisinart CPT-122 Compact Plastic 2-Slice Toas...,55.462963,0.460000,29.95,1,0,4.3,27270,https://www.amazon.com/stores/Cuisinart/page/9...,...,2018-01-01,1,.,0,6628,24,"Very happy. Thanks, Tom.",0.6,positive,0.765
3,B0744M3SB4,Nostalgia TCS2 Grilled Cheese Toaster with Eas...,209.988477,0.786655,44.8,1,0,4.1,4156,https://www.amazon.com/stores/Nostalgia/page/B...,...,2018-01-01,1,12,1,863,413,"I bought this for me husband for Christmas, af...",0.508333,positive,0.4754
4,B07H81RZ9Q,Hamilton Beach 2 Slice Extra Wide Slot Toaster...,58.890000,0.000000,58.89,1,0,4.2,9529,https://www.amazon.com/stores/HamiltonBeach/pa...,...,2018-01-01,1,1,0,4979,106,"Worked OK, never above average. Died one year ...",0.45,negative,-0.6908


In [4]:
toaster_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 85262 entries, 0 to 85261
Data columns (total 30 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ASIN         85262 non-null  str           
 1   P_TITLE      85262 non-null  str           
 2   OP           67023 non-null  float64       
 3   DP           85262 non-null  float64       
 4   SP           85262 non-null  object        
 5   FS           85262 non-null  object        
 6   PRA_4.5      85262 non-null  int64         
 7   P_RTG        85262 non-null  object        
 8   RTG_P_NO     85262 non-null  object        
 9   SELLER_LINK  85262 non-null  str           
 10  IMAGE_URL    85262 non-null  str           
 11  P_URL        85262 non-null  str           
 12  RV_URL       85262 non-null  str           
 13  PRFL_IMG     85262 non-null  str           
 14  PRFL_URL     85262 non-null  str           
 15  RV_TTL       85259 non-null  str           
 16  RVS          85

In [5]:
# Coerce Review_Date to datetime
toaster_df["RV_DT"] = pd.to_datetime(toaster_df["RV_DT"], errors="coerce")

# Coerce numeric columns to numeric
num_cols = [
    "OP", "DP", "SP", "FS", "PRA_4.5", "P_RTG", "RTG_P_NO",
    "RSR", "VP", "HLP_VT", "IMG_PRST", "TTL_RV", "RVS_L",
    "SUBJ", "CP_RVS"]
    
for col in num_cols:
    toaster_df[col] = pd.to_numeric(toaster_df[col], errors="coerce") 

In [6]:
# Size before deduplication
print(f"Dataset size before deduplication: {toaster_df.shape[0]} rows")

# Remove duplicate rows by keeping the first date (RV_DT) for each unique combination of reviewer (RVR) and product (ASIN)
toaster_df = (
    toaster_df.sort_values(["RVR", "ASIN", "RV_DT"], ascending=[True, True, False])
    .drop_duplicates(subset=["RVR", "ASIN"], keep="first")
    .reset_index(drop=True)
)

# Size after deduplication
print(f"Dataset size after deduplication: {toaster_df.shape[0]} rows")

Dataset size before deduplication: 85262 rows
Dataset size after deduplication: 62014 rows


In [7]:
# Remove reviews with missing ratings (RSR) and text (RVS)
toaster_df = toaster_df.dropna(subset=["RVS", "RSR"])

# Remove short reviews (less than 10 characters)
toaster_df = toaster_df[toaster_df["RVS_L"] > 10].copy()

# Remove products with less than 3 reviews
toaster_df = toaster_df.groupby("ASIN").filter(lambda x: len(x) >= 3)

# Remove emoji-only reviews (reviews where the text contains only emojis)
toaster_df = toaster_df[~toaster_df["RVS"].str.contains(r"^\s*[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+\s*$", regex=True)].copy() 

# Size after cleaning
print(f"Dataset size after cleaning: {toaster_df.shape[0]} rows")

Dataset size after cleaning: 59586 rows


# Save Cleaned Data

In [8]:
toaster_df.to_csv(os.path.join(data_processed_dir, "toaster_clean.csv"), index=False)